# Movie Rental Data Warehouse ETL — Real Sakila Database

This notebook implements the ETL process for a Movie Rental Data Warehouse using the real `sakila` OLTP database as the source system.

The original `sakila` database is designed for daily transaction processing, such as recording rentals, payments, customers, films, inventory, staff, stores, and location information. However, this OLTP structure is not optimized for analytical reporting. Therefore, this notebook transforms the operational data into a dimensional data warehouse model that supports business analysis and decision-making.

The implemented data warehouse focuses on two main business processes:

1. **Rental transactions**
2. **Payment transactions**

These processes are modeled using fact tables connected to shared dimension tables, allowing analysis of rental activity, revenue, customer behavior, film popularity, store performance, staff performance, and trends over time.


## ETL Steps

1. **Extract** source tables from the `sakila` OLTP database.
2. **Transform** operational data into dimension and fact tables.
3. **Load** the transformed tables into the target `movie_rental_dw` database.
4. **Validate** the loaded warehouse tables using data quality checks and row-count comparisons.

## Output Data Warehouse Tables

### Dimension Tables

- `dim_date`
- `dim_location`
- `dim_customer`
- `dim_film`
- `dim_store`
- `dim_staff`

### Fact Tables

- `fact_rental`
- `fact_payment`

## 1. Install Required Libraries

In [3]:
%pip install pandas numpy sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

In [5]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass
from sqlalchemy.engine import URL


## 3. Create Database Connections

Connects to:
- Source OLTP database: `sakila`
- Target Data Warehouse database: `movie_rental_dw`


In [7]:
MYSQL_USER = "root"
MYSQL_PASSWORD = "root"
MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
SOURCE_DB = "sakila"
TARGET_DB = "movie_rental_dw"

server_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT
)

server_engine = create_engine(server_url)

with server_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {TARGET_DB}"))
    conn.commit()

source_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=SOURCE_DB
)

dw_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=TARGET_DB
)

source_engine = create_engine(source_url)
dw_engine = create_engine(dw_url)

print("Connections created successfully.")
print(f"Source database: {SOURCE_DB}")
print(f"Target database: {TARGET_DB}")


Connections created successfully.
Source database: sakila
Target database: movie_rental_dw


## 4. Test Connections

In [9]:
with source_engine.connect() as conn:
    rental_count = conn.execute(text("SELECT COUNT(*) FROM rental")).fetchone()[0]
    payment_count = conn.execute(text("SELECT COUNT(*) FROM payment")).fetchone()[0]

print("Sakila connected successfully.")
print("Rental rows:", rental_count)
print("Payment rows:", payment_count)

Sakila connected successfully.
Rental rows: 16044
Payment rows: 16044


---
# Extract Phase

Extract all required OLTP tables from `sakila`.

In [11]:
tables = [
    "rental",
    "payment",
    "customer",
    "film",
    "inventory",
    "store",
    "staff",
    "address",
    "city",
    "country",
    "category",
    "film_category",
    "language",
    "actor",
    "film_actor"
]

data = {}

for table in tables:
    data[table] = pd.read_sql(f"SELECT * FROM {table}", source_engine)
    print(f"Loaded {table}: {data[table].shape}")

rental       = data["rental"]
payment      = data["payment"]
customer     = data["customer"]
film         = data["film"]
inventory    = data["inventory"]
store        = data["store"]
staff        = data["staff"]
address      = data["address"]
city         = data["city"]
country      = data["country"]
category     = data["category"]
film_category = data["film_category"]
language     = data["language"]
actor        = data["actor"]
film_actor   = data["film_actor"]

Loaded rental: (16044, 7)
Loaded payment: (16044, 7)
Loaded customer: (599, 9)
Loaded film: (1000, 13)
Loaded inventory: (4581, 4)
Loaded store: (2, 4)
Loaded staff: (2, 11)
Loaded address: (603, 9)
Loaded city: (600, 4)
Loaded country: (109, 3)
Loaded category: (16, 3)
Loaded film_category: (1000, 3)
Loaded language: (6, 3)
Loaded actor: (200, 4)
Loaded film_actor: (5462, 3)


## Data Understanding

In [13]:
print("Rental shape:", rental.shape)
display(rental.head())

print("Payment shape:", payment.shape)
display(payment.head())

print("Customer shape:", customer.shape)
display(customer.head())

print("Film shape:", film.shape)
display(film.head())

Rental shape: (16044, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


Payment shape: (16044, 7)


,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76,2.99,2005-05-25 11:30:37,2006-02-15 22:12:30
1,2,1,1,573,0.99,2005-05-28 10:35:23,2006-02-15 22:12:30
2,3,1,1,1185,5.99,2005-06-15 00:54:12,2006-02-15 22:12:30
3,4,1,2,1422,0.99,2005-06-15 18:02:53,2006-02-15 22:12:30
4,5,1,2,1476,9.99,2005-06-15 21:08:46,2006-02-15 22:12:30


Customer shape: (599, 9)


,customer_id,store_id,first_name,last_name,email,address_id,active,create_date,last_update
0,1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,1,2006-02-14 22:04:36,2006-02-15 04:57:20
1,2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,1,2006-02-14 22:04:36,2006-02-15 04:57:20
2,3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,1,2006-02-14 22:04:36,2006-02-15 04:57:20
3,4,2,BARBARA,JONES,BARBARA.JONES@sakilacustomer.org,8,1,2006-02-14 22:04:36,2006-02-15 04:57:20
4,5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,1,2006-02-14 22:04:36,2006-02-15 04:57:20


Film shape: (1000, 13)


,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


---
# Transform Phase

Create dimension and fact tables from the OLTP source tables.

## 5. Data Cleaning

In [15]:
# Check duplicates
print("=== Duplicate Rows ===")
for name in data:
    duplicates = data[name].duplicated().sum()
    print(f"{name}: {duplicates} duplicate rows")

=== Duplicate Rows ===
rental: 0 duplicate rows
payment: 0 duplicate rows
customer: 0 duplicate rows
film: 0 duplicate rows
inventory: 0 duplicate rows
store: 0 duplicate rows
staff: 0 duplicate rows
address: 0 duplicate rows
city: 0 duplicate rows
country: 0 duplicate rows
category: 0 duplicate rows
film_category: 0 duplicate rows
language: 0 duplicate rows
actor: 0 duplicate rows
film_actor: 0 duplicate rows


In [16]:
# Check nulls
print("=== Null Values ===")
for name in data:
    nulls = data[name].isnull().sum()
    if nulls.any():
        print(f"\n{name}")
        print(nulls[nulls > 0])

=== Null Values ===

rental
return_date    183
dtype: int64

film
original_language_id    1000
dtype: int64

staff
picture     1
password    1
dtype: int64

address
address2    4
dtype: int64


## Missing Values Explanation

The null-value check shows that only a few source columns contain missing values.

These missing values are expected in the Sakila database and do not cause a problem for the data warehouse:

- `rental.return_date` has missing values because some films have not been returned yet.
- `film.original_language_id` is missing for all films, but this column is not needed because the warehouse uses `language_id`.
- `staff.picture` and `staff.password` are not used in the data warehouse.
- `address.address2` is optional, so missing values are normal.

During the transformation step, important missing values are handled properly. For example, unreturned rentals are marked with `rental_duration_days = -1`, and optional address fields are replaced with `"Unknown"` in the location dimension.

Therefore, the missing values found in the source data are acceptable and do not affect the correctness of the warehouse.

In [18]:
# Remove duplicates from all tables
for name in data:
    before = len(data[name])
    data[name] = data[name].drop_duplicates()
    after = len(data[name])
    if before != after:
        print(f"{name}: removed {before - after} duplicate rows")

rental        = data["rental"]
payment       = data["payment"]
customer      = data["customer"]
film          = data["film"]
inventory     = data["inventory"]
store         = data["store"]
staff         = data["staff"]
address       = data["address"]
city          = data["city"]
country       = data["country"]
category      = data["category"]
film_category = data["film_category"]
language      = data["language"]
actor         = data["actor"]
film_actor    = data["film_actor"]

print("Data cleaning completed.")

Data cleaning completed.


## 6. Create Dim_Date

In [20]:
all_dates = pd.concat([
    pd.to_datetime(rental["rental_date"], errors="coerce"),
    pd.to_datetime(rental["return_date"], errors="coerce"),
    pd.to_datetime(payment["payment_date"], errors="coerce")
]).dropna().drop_duplicates()

dim_date = pd.DataFrame({"full_date": all_dates})
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"]).dt.date
dim_date = dim_date.drop_duplicates()

dim_date["date_key"]    = pd.to_datetime(dim_date["full_date"]).dt.strftime("%Y%m%d").astype(int)
dim_date["day"]         = pd.to_datetime(dim_date["full_date"]).dt.day
dim_date["month"]       = pd.to_datetime(dim_date["full_date"]).dt.month
dim_date["month_name"]  = pd.to_datetime(dim_date["full_date"]).dt.month_name()
dim_date["quarter"]     = pd.to_datetime(dim_date["full_date"]).dt.quarter
dim_date["year"]        = pd.to_datetime(dim_date["full_date"]).dt.year
dim_date["day_of_week"] = pd.to_datetime(dim_date["full_date"]).dt.day_name()

# Unknown date row — used for unreturned films (return_date IS NULL)
unknown_date = pd.DataFrame({
    "date_key":    [19000101],
    "full_date":   [pd.to_datetime("1900-01-01").date()],
    "day":         [1],
    "month":       [1],
    "month_name":  ["January"],
    "quarter":     [1],
    "year":        [1900],
    "day_of_week": ["Monday"]
})

dim_date = pd.concat([unknown_date, dim_date], ignore_index=True)
dim_date = dim_date[
    ["date_key", "full_date", "day", "month", "month_name", "quarter", "year", "day_of_week"]
].drop_duplicates("date_key")

display(dim_date.head())
print("Dim_Date rows:", len(dim_date))

,date_key,full_date,day,month,month_name,quarter,year,day_of_week
0,19000101,1900-01-01,1,1,January,1,1900,Monday
1,20050524,2005-05-24,24,5,May,2,2005,Tuesday
2,20050525,2005-05-25,25,5,May,2,2005,Wednesday
3,20050526,2005-05-26,26,5,May,2,2005,Thursday
4,20050527,2005-05-27,27,5,May,2,2005,Friday


Dim_Date rows: 91


## 7. Create Dim_Location

In [22]:
location_full = address.merge(city, on="city_id", how="left")
location_full = location_full.merge(country, on="country_id", how="left")

dim_location = location_full[[
    "address_id",
    "address",
    "address2",
    "district",
    "city",
    "country",
    "postal_code",
    "phone"
]].drop_duplicates().reset_index(drop=True)

dim_location["address2"] = dim_location["address2"].fillna("Unknown")
dim_location["postal_code"] = dim_location["postal_code"].fillna("Unknown")
dim_location["phone"] = dim_location["phone"].fillna("Unknown")

dim_location.insert(0, "location_key", range(1, len(dim_location) + 1))

display(dim_location.head())
print("Dim_Location rows:", len(dim_location))


,location_key,address_id,address,address2,district,city,country,postal_code,phone
0,1,1,47 MySakila Drive,Unknown,Alberta,Lethbridge,Canada,,
1,2,2,28 MySQL Boulevard,Unknown,QLD,Woodridge,Australia,,
2,3,3,23 Workhaven Lane,Unknown,Alberta,Lethbridge,Canada,,14033335568
3,4,4,1411 Lillydale Drive,Unknown,QLD,Woodridge,Australia,,6172235589
4,5,5,1913 Hanoi Way,,Nagasaki,Sasebo,Japan,35200,28303384290


Dim_Location rows: 603


## 8. Create Dim_Customer

In [24]:
dim_customer = customer.merge(
    dim_location[["location_key", "address_id"]],
    on="address_id",
    how="left"
)

dim_customer["customer_name"] = (
    dim_customer["first_name"].fillna("") + " " + dim_customer["last_name"].fillna("")
).str.strip()

dim_customer["email"] = dim_customer["email"].fillna("Unknown")

dim_customer = dim_customer[[
    "customer_id",
    "customer_name",
    "email",
    "active",
    "create_date",
    "location_key"
]].drop_duplicates().reset_index(drop=True)

dim_customer.insert(0, "customer_key", range(1, len(dim_customer) + 1))

display(dim_customer.head())
print("Dim_Customer rows:", len(dim_customer))

,customer_key,customer_id,customer_name,email,active,create_date,location_key
0,1,1,MARY SMITH,MARY.SMITH@sakilacustomer.org,1,2006-02-14 22:04:36,5
1,2,2,PATRICIA JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,1,2006-02-14 22:04:36,6
2,3,3,LINDA WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,1,2006-02-14 22:04:36,7
3,4,4,BARBARA JONES,BARBARA.JONES@sakilacustomer.org,1,2006-02-14 22:04:36,8
4,5,5,ELIZABETH BROWN,ELIZABETH.BROWN@sakilacustomer.org,1,2006-02-14 22:04:36,9


Dim_Customer rows: 599


## 9. Create Dim_Store

In [26]:
# Build manager name from staff table
manager_info = staff.copy()
manager_info["manager_name"] = (
    manager_info["first_name"].fillna("") + " " + manager_info["last_name"].fillna("")
).str.strip()
manager_info = manager_info[["staff_id", "manager_name"]]

dim_store = store.merge(
    dim_location[["location_key", "address_id"]],
    on="address_id",
    how="left"
)

# Join manager name using manager_staff_id
dim_store = dim_store.merge(
    manager_info,
    left_on="manager_staff_id",
    right_on="staff_id",
    how="left"
)

dim_store = dim_store[[
    "store_id",
    "manager_staff_id",
    "manager_name",
    "location_key"
]].drop_duplicates().reset_index(drop=True)

dim_store.insert(0, "store_key", range(1, len(dim_store) + 1))

display(dim_store.head())
print("Dim_Store rows:", len(dim_store))

,store_key,store_id,manager_staff_id,manager_name,location_key
0,1,1,1,Mike Hillyer,1
1,2,2,2,Jon Stephens,2


Dim_Store rows: 2


## 10. Create Dim_Staff

In [28]:
dim_staff = staff.merge(
    dim_store[["store_key", "store_id"]],
    on="store_id",
    how="left"
)

dim_staff = dim_staff.merge(
    dim_location[["location_key", "address_id"]],
    on="address_id",
    how="left"
)

dim_staff["staff_name"] = (
    dim_staff["first_name"].fillna("") + " " + dim_staff["last_name"].fillna("")
).str.strip()

dim_staff["email"] = dim_staff["email"].fillna("Unknown")

dim_staff = dim_staff[[
    "staff_id",
    "staff_name",
    "email",
    "active",
    "username",
    "store_key",
    "location_key"
]].drop_duplicates().reset_index(drop=True)

dim_staff.insert(0, "staff_key", range(1, len(dim_staff) + 1))

display(dim_staff.head())
print("Dim_Staff rows:", len(dim_staff))

,staff_key,staff_id,staff_name,email,active,username,store_key,location_key
0,1,1,Mike Hillyer,Mike.Hillyer@sakilastaff.com,1,Mike,1,3
1,2,2,Jon Stephens,Jon.Stephens@sakilastaff.com,1,Jon,2,4


Dim_Staff rows: 2


## 11. Create Dim_Film

In [30]:
# Build actors string per film
actor_full = film_actor.merge(actor, on="actor_id", how="left")
actor_full["actor_name"] = (
    actor_full["first_name"].fillna("") + " " + actor_full["last_name"].fillna("")
).str.strip()

film_actors_grouped = (
    actor_full.groupby("film_id")["actor_name"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
    .rename(columns={"actor_name": "actors"})
)

film_cat_full = film_category.merge(category[["category_id", "name"]], on="category_id", how="left")
film_categories_grouped = (
    film_cat_full.groupby("film_id")["name"]
    .apply(lambda x: ", ".join(sorted(set(x))))
    .reset_index()
    .rename(columns={"name": "categories"})
)

# Build film_full — start from film, join language, categories, actors
film_full = film.merge(language[["language_id", "name"]], on="language_id", how="left")
film_full = film_full.rename(columns={"name": "language"})

film_full = film_full.merge(film_categories_grouped, on="film_id", how="left")
film_full = film_full.merge(film_actors_grouped, on="film_id", how="left")

dim_film = film_full[[
    "film_id",
    "title",
    "description",
    "release_year",
    "language",
    "categories",
    "rental_duration",
    "rental_rate",
    "length",
    "replacement_cost",
    "rating",
    "special_features",
    "actors"
]].drop_duplicates("film_id").reset_index(drop=True)

dim_film.insert(0, "film_key", range(1, len(dim_film) + 1))

display(dim_film.head())
print("Dim_Film rows:", len(dim_film))

,film_key,film_id,title,description,release_year,language,categories,rental_duration,rental_rate,length,replacement_cost,rating,special_features,actors
0,1,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,English,Documentary,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes","CHRISTIAN GABLE, JOHNNY CAGE, LUCILLE TRACY, M..."
1,2,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,English,Horror,3,4.99,48,12.99,G,"Trailers,Deleted Scenes","BOB FAWCETT, CHRIS DEPP, MINNIE ZELLWEGER, SEA..."
2,3,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,English,Documentary,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes","BOB FAWCETT, CAMERON STREEP, JULIANNE DENCH, N..."
3,4,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,English,Horror,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes","FAY WINSLET, JODIE DEGENERES, KENNETH PESCI, O..."
4,5,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,English,Family,6,2.99,130,22.99,G,Deleted Scenes,"DUSTIN TAUTOU, GARY PHOENIX, MATTHEW CARREY, M..."


Dim_Film rows: 1000


## 12. Create Fact_Rental

Grain: **one row per rental transaction**.

In [32]:
fact_rental = rental.merge(
    inventory[["inventory_id", "film_id", "store_id"]],
    on="inventory_id",
    how="left"
)

# Date keys
fact_rental["rental_date_key"] = pd.to_datetime(
    fact_rental["rental_date"], errors="coerce"
).dt.strftime("%Y%m%d").astype(int)

fact_rental["return_date_key"] = pd.to_datetime(
    fact_rental["return_date"], errors="coerce"
).dt.strftime("%Y%m%d")

# NULL return_date → film not yet returned → map to unknown date key 19000101
fact_rental["return_date_key"] = fact_rental["return_date_key"].fillna("19000101").astype(int)

# --- FIX: Use -1 for unreturned films instead of 0 ---
# 0 is ambiguous (could mean returned same day). -1 clearly means "not yet returned".
fact_rental["rental_duration_days"] = (
    pd.to_datetime(fact_rental["return_date"], errors="coerce")
    - pd.to_datetime(fact_rental["rental_date"], errors="coerce")
).dt.days

fact_rental["rental_duration_days"] = fact_rental["rental_duration_days"].fillna(-1).astype(int)

fact_rental["rental_count"] = 1

# Join surrogate keys
fact_rental = fact_rental.merge(
    dim_customer[["customer_key", "customer_id"]],
    on="customer_id",
    how="left"
)

fact_rental = fact_rental.merge(
    dim_film[["film_key", "film_id", "rental_duration"]],
    on="film_id",
    how="left"
)

fact_rental = fact_rental.merge(
    dim_store[["store_key", "store_id"]],
    on="store_id",
    how="left"
)

fact_rental = fact_rental.merge(
    dim_staff[["staff_key", "staff_id"]],
    on="staff_id",
    how="left"
)

# Late return calculation — only for returned films (rental_duration_days >= 0)
fact_rental["late_days"] = fact_rental.apply(
    lambda row: max(row["rental_duration_days"] - row["rental_duration"], 0)
    if row["rental_duration_days"] >= 0 else 0,
    axis=1
)

fact_rental["late_return_flag"] = fact_rental["late_days"].apply(lambda x: 1 if x > 0 else 0)

fact_rental_final = fact_rental[[
    "rental_id",
    "rental_date_key",
    "return_date_key",
    "customer_key",
    "film_key",
    "store_key",
    "staff_key",
    "rental_count",
    "rental_duration_days",
    "late_return_flag",
    "late_days"
]].copy()

fact_rental_final.insert(0, "rental_fact_key", range(1, len(fact_rental_final) + 1))

display(fact_rental_final.head())
print("Fact_Rental rows:", len(fact_rental_final))

,rental_fact_key,rental_id,rental_date_key,return_date_key,customer_key,film_key,store_key,staff_key,rental_count,rental_duration_days,late_return_flag,late_days
0,1,1,20050524,20050526,130,80,1,1,1,1,0,0
1,2,2,20050524,20050528,459,333,2,1,1,3,0,0
2,3,3,20050524,20050601,408,373,2,1,1,7,0,0
3,4,4,20050524,20050603,333,535,1,2,1,9,1,3
4,5,5,20050524,20050602,222,450,2,1,1,8,1,3


Fact_Rental rows: 16044


## 13. Create Fact_Payment

Grain: **one row per payment transaction**.

In [34]:
fact_payment = payment.copy()

fact_payment["payment_date_key"] = pd.to_datetime(
    fact_payment["payment_date"], errors="coerce"
).dt.strftime("%Y%m%d").astype(int)

fact_payment["payment_count"] = 1

# Join rental to get inventory_id
fact_payment = fact_payment.merge(
    rental[["rental_id", "inventory_id"]],
    on="rental_id",
    how="left"
)

# Join inventory to get store_id and film_id
fact_payment = fact_payment.merge(
    inventory[["inventory_id", "store_id", "film_id"]],
    on="inventory_id",
    how="left"
)

# Join surrogate keys
fact_payment = fact_payment.merge(
    dim_customer[["customer_key", "customer_id"]],
    on="customer_id",
    how="left"
)

fact_payment = fact_payment.merge(
    dim_staff[["staff_key", "staff_id"]],
    on="staff_id",
    how="left"
)

fact_payment = fact_payment.merge(
    dim_store[["store_key", "store_id"]],
    on="store_id",
    how="left"
)

# --- FIX: Add film_key to fact_payment ---
# This allows direct revenue analysis by film without going through fact_rental.
fact_payment = fact_payment.merge(
    dim_film[["film_key", "film_id"]],
    on="film_id",
    how="left"
)

fact_payment_final = fact_payment[[
    "payment_id",
    "payment_date_key",
    "customer_key",
    "staff_key",
    "store_key",
    "film_key",
    "amount",
    "payment_count"
]].copy()

fact_payment_final = fact_payment_final.rename(columns={"amount": "payment_amount"})
fact_payment_final.insert(0, "payment_fact_key", range(1, len(fact_payment_final) + 1))

display(fact_payment_final.head())
print("Fact_Payment rows:", len(fact_payment_final))

,payment_fact_key,payment_id,payment_date_key,customer_key,staff_key,store_key,film_key,payment_amount,payment_count
0,1,1,20050525,1,1,2,663,2.99,1
1,2,2,20050528,1,1,2,875,0.99,1
2,3,3,20050615,1,1,1,611,5.99,1
3,4,4,20050615,1,2,2,228,0.99,1
4,5,5,20050615,1,2,1,308,9.99,1


Fact_Payment rows: 16044


---
# Data Quality Checks



In [36]:
print("=== Data Quality Checks ===")

# --- fact_rental checks ---
missing_customer = fact_rental_final["customer_key"].isnull().sum()
missing_film     = fact_rental_final["film_key"].isnull().sum()
missing_store    = fact_rental_final["store_key"].isnull().sum()
missing_staff    = fact_rental_final["staff_key"].isnull().sum()

print(f"fact_rental — missing customer_key : {missing_customer}")
print(f"fact_rental — missing film_key     : {missing_film}")
print(f"fact_rental — missing store_key    : {missing_store}")
print(f"fact_rental — missing staff_key    : {missing_staff}")

assert missing_customer == 0, "fact_rental has rows with missing customer_key!"
assert missing_film     == 0, "fact_rental has rows with missing film_key!"
assert missing_store    == 0, "fact_rental has rows with missing store_key!"
assert missing_staff    == 0, "fact_rental has rows with missing staff_key!"

# --- fact_payment checks ---
missing_pay_customer = fact_payment_final["customer_key"].isnull().sum()
missing_pay_staff    = fact_payment_final["staff_key"].isnull().sum()
missing_pay_store    = fact_payment_final["store_key"].isnull().sum()
missing_pay_film     = fact_payment_final["film_key"].isnull().sum()
negative_amount      = (fact_payment_final["payment_amount"] < 0).sum()

print(f"fact_payment — missing customer_key : {missing_pay_customer}")
print(f"fact_payment — missing staff_key    : {missing_pay_staff}")
print(f"fact_payment — missing store_key    : {missing_pay_store}")
print(f"fact_payment — missing film_key     : {missing_pay_film}")
print(f"fact_payment — negative amounts     : {negative_amount}")

assert missing_pay_customer == 0, "fact_payment has rows with missing customer_key!"
assert missing_pay_staff    == 0, "fact_payment has rows with missing staff_key!"
assert missing_pay_store    == 0, "fact_payment has rows with missing store_key!"
assert missing_pay_film     == 0, "fact_payment has rows with missing film_key!"
assert negative_amount      == 0, "fact_payment has negative payment amounts!"

# --- dimension key checks ---
assert dim_date["date_key"].duplicated().sum() == 0, "dim_date has duplicate date_keys!"
assert dim_location["location_key"].duplicated().sum() == 0, "dim_location has duplicate location_keys!"
assert dim_customer["customer_key"].duplicated().sum() == 0, "dim_customer has duplicate customer_keys!"
assert dim_store["store_key"].duplicated().sum() == 0, "dim_store has duplicate store_keys!"
assert dim_staff["staff_key"].duplicated().sum() == 0, "dim_staff has duplicate staff_keys!"
assert dim_film["film_key"].duplicated().sum() == 0, "dim_film has duplicate film_keys!"

assert dim_customer["customer_id"].duplicated().sum() == 0, "dim_customer has duplicate customer_ids!"
assert dim_film["film_id"].duplicated().sum() == 0, "dim_film has duplicate film_ids!"

print("\n All data quality checks passed.")

=== Data Quality Checks ===
fact_rental — missing customer_key : 0
fact_rental — missing film_key     : 0
fact_rental — missing store_key    : 0
fact_rental — missing staff_key    : 0
fact_payment — missing customer_key : 0
fact_payment — missing staff_key    : 0
fact_payment — missing store_key    : 0
fact_payment — missing film_key     : 0
fact_payment — negative amounts     : 0

 All data quality checks passed.


---
# Load Phase

Dimension tables are loaded **before** fact tables because facts reference dimension surrogate keys.

In [38]:
# Load dimensions first
# Important: to_sql(if_exists="replace") recreates the tables, so PKs must be added after loading.
dim_date.to_sql("dim_date",         dw_engine, if_exists="replace", index=False)
dim_location.to_sql("dim_location", dw_engine, if_exists="replace", index=False)
dim_customer.to_sql("dim_customer", dw_engine, if_exists="replace", index=False)
dim_store.to_sql("dim_store",       dw_engine, if_exists="replace", index=False)
dim_staff.to_sql("dim_staff",       dw_engine, if_exists="replace", index=False)
dim_film.to_sql("dim_film",         dw_engine, if_exists="replace", index=False)

# Load facts after dimensions
fact_rental_final.to_sql("fact_rental",   dw_engine, if_exists="replace", index=False)
fact_payment_final.to_sql("fact_payment", dw_engine, if_exists="replace", index=False)

# Add primary keys after pandas creates the tables.
# This makes the physical DW tables match the dimensional model documentation.
primary_key_sql = [
    "ALTER TABLE dim_date ADD PRIMARY KEY (date_key)",
    "ALTER TABLE dim_location ADD PRIMARY KEY (location_key)",
    "ALTER TABLE dim_customer ADD PRIMARY KEY (customer_key)",
    "ALTER TABLE dim_store ADD PRIMARY KEY (store_key)",
    "ALTER TABLE dim_staff ADD PRIMARY KEY (staff_key)",
    "ALTER TABLE dim_film ADD PRIMARY KEY (film_key)",
    "ALTER TABLE fact_rental ADD PRIMARY KEY (rental_fact_key)",
    "ALTER TABLE fact_payment ADD PRIMARY KEY (payment_fact_key)"
]

with dw_engine.connect() as conn:
    for sql in primary_key_sql:
        conn.execute(text(sql))
    conn.commit()

print("Load phase completed successfully.")
print("Primary keys added successfully.")
print("Data Warehouse tables loaded into movie_rental_dw.")

Load phase completed successfully.
Primary keys added successfully.
Data Warehouse tables loaded into movie_rental_dw.


## 14. Verify Data Warehouse Tables

In [40]:
dw_tables = pd.read_sql("SHOW TABLES", dw_engine)
display(dw_tables)

row_counts = {}

for table in ["dim_date", "dim_location", "dim_customer", "dim_film",
              "dim_store", "dim_staff", "fact_rental", "fact_payment"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table}", dw_engine)
    row_counts[table] = int(count.loc[0, "row_count"])
    print(table, ":", row_counts[table], "rows")

# Final validation: make sure the DW did not lose source transactions.
assert row_counts["dim_film"] == len(film), "dim_film row count does not match source film table!"
assert row_counts["fact_rental"] == len(rental), "fact_rental row count does not match source rental table!"
assert row_counts["fact_payment"] == len(payment), "fact_payment row count does not match source payment table!"

print("\nFinal row-count validation passed.")

,Tables_in_movie_rental_dw
0,dim_customer
1,dim_date
2,dim_film
3,dim_location
4,dim_staff
5,dim_store
6,fact_payment
7,fact_rental


dim_date : 91 rows
dim_location : 603 rows
dim_customer : 599 rows
dim_film : 1000 rows
dim_store : 2 rows
dim_staff : 2 rows
fact_rental : 16044 rows
fact_payment : 16044 rows

Final row-count validation passed.


---
## 15. Example Analytical Queries

In [42]:
# Query 1: Top 10 most rented films
top_rented_films = pd.read_sql("""
SELECT 
    f.title,
    f.categories,
    COUNT(r.rental_fact_key) AS total_rentals
FROM fact_rental r
JOIN dim_film f ON r.film_key = f.film_key
GROUP BY f.title, f.categories
ORDER BY total_rentals DESC
LIMIT 10;
""", dw_engine)

print("=== Top 10 Most Rented Films ===")
display(top_rented_films)

=== Top 10 Most Rented Films ===


,title,categories,total_rentals
0,BUCKET BROTHERHOOD,Travel,34
1,ROCKETEER MOTHER,Foreign,33
2,RIDGEMONT SUBMARINE,New,32
3,SCALAWAG DUCK,Music,32
4,GRIT CLOCKWORK,Games,32
5,FORWARD TEMPLE,Games,32
6,JUGGLER HARDLY,Animation,32
7,TIMBERLAND SKY,Classics,31
8,ZORRO ARK,Comedy,31
9,NETWORK PEAK,Family,31


In [43]:
# Query 2: Revenue by month
revenue_by_month = pd.read_sql("""
SELECT 
    d.year,
    d.month,
    d.month_name,
    SUM(p.payment_amount) AS total_revenue
FROM fact_payment p
JOIN dim_date d ON p.payment_date_key = d.date_key
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month;
""", dw_engine)

print("=== Revenue by Month ===")
display(revenue_by_month)

=== Revenue by Month ===


,year,month,month_name,total_revenue
0,2005,5,May,4823.44
1,2005,6,June,9629.89
2,2005,7,July,28368.91
3,2005,8,August,24070.14
4,2006,2,February,514.18


In [44]:
# Query 3: Store performance by rentals
store_performance = pd.read_sql("""
SELECT 
    s.store_id,
    s.manager_name,
    COUNT(r.rental_fact_key) AS total_rentals
FROM fact_rental r
JOIN dim_store s ON r.store_key = s.store_key
GROUP BY s.store_id, s.manager_name
ORDER BY total_rentals DESC;
""", dw_engine)

print("=== Store Performance ===")
display(store_performance)

=== Store Performance ===


,store_id,manager_name,total_rentals
0,2,Jon Stephens,8121
1,1,Mike Hillyer,7923


In [45]:
# Query 4: Revenue by film (NEW — only possible now that film_key is in fact_payment)
revenue_by_film = pd.read_sql("""
SELECT 
    f.title,
    f.categories,
    SUM(p.payment_amount) AS total_revenue,
    COUNT(p.payment_fact_key) AS total_payments
FROM fact_payment p
JOIN dim_film f ON p.film_key = f.film_key
GROUP BY f.title, f.categories
ORDER BY total_revenue DESC
LIMIT 10;
""", dw_engine)

print("=== Top 10 Films by Revenue ===")
display(revenue_by_film)

=== Top 10 Films by Revenue ===


,title,categories,total_revenue,total_payments
0,TELEGRAPH VOYAGE,Music,231.73,27
1,WIFE TURN,Documentary,223.69,31
2,ZORRO ARK,Comedy,214.69,31
3,GOODFELLAS SALUTE,Sci-Fi,209.69,31
4,SATURDAY LAMBS,Sports,204.72,28
5,TITANS JERK,Sci-Fi,201.71,29
6,TORQUE BOUND,Drama,198.72,27
7,HARRY IDAHO,Drama,195.70,30
8,INNOCENT USUAL,Foreign,191.74,26
9,HUSTLER PARTY,Comedy,190.78,22


In [46]:
# Query 5: Late return analysis
late_returns = pd.read_sql("""
SELECT 
    f.title,
    f.categories,
    COUNT(*) AS late_return_count,
    AVG(r.late_days) AS avg_late_days
FROM fact_rental r
JOIN dim_film f ON r.film_key = f.film_key
WHERE r.late_return_flag = 1
GROUP BY f.title, f.categories
ORDER BY late_return_count DESC
LIMIT 10;
""", dw_engine)

print("=== Top 10 Films with Most Late Returns ===")
display(late_returns)

=== Top 10 Films with Most Late Returns ===


,title,categories,late_return_count,avg_late_days
0,RIDGEMONT SUBMARINE,New,24,3.6250
1,BUTTERFLY CHOCOLAT,New,23,3.3478
2,TELEGRAPH VOYAGE,Music,22,3.7727
3,TIMBERLAND SKY,Classics,21,3.9048
4,CHANCE RESURRECTION,Sports,20,2.4500
5,GRIT CLOCKWORK,Games,20,3.4000
6,ROCKETEER MOTHER,Foreign,20,3.7500
7,ENGLISH BULWORTH,Sci-Fi,20,3.0500
8,SATURDAY LAMBS,Sports,19,2.8421
9,PRINCESS GIANT,Documentary,19,2.9474
